This file turns the 2d correlations into projection plots, and finds v2

In [1]:
import ROOT
import numpy as np
import math
import pandas as pd
import fastjet
import matplotlib.pyplot as plt
import os
import ctypes
import array

In [2]:
ROOT.gDirectory.Clear()

In [3]:
#output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/dataset1_2pc_output_histograms"
#output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/dataset2_2pc_output_histograms"
#output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/batch0"
#output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/batch1"
#output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/batch2"

In [4]:
#output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/batch0"
#output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/batch1"
#output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/merged"


In [21]:
output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/0mb_nch60"

In [23]:
output_dir = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_nch60"

In [24]:
#read the files

def pathfinder(mult_bin):
    '''
    gets the file path for a given mult bin
    input: multiplicity bin
    outputs:
        - yield histograms root file path
        - yield histogram wta title
        - yield histogram std title
    '''
    # 1. Define your specific output folder path

    # Create the folder if it doesn't already exist
    os.makedirs(output_dir, exist_ok=True)

    # 2. Create the dynamic filename
    if len(mult_bin) == 2:
        bin_name = f"{mult_bin[0]}_{mult_bin[1]}"
    else:
        bin_name = f"{mult_bin[0]}_up"

    #filename = f"Analysis_Output_batch1.root"
    filename = f"merged_yields.root"

    # 3. Combine the folder path and the filename
    full_filepath = os.path.join(output_dir, filename)

    #wta_title = f"hYield_WTA_{bin_name}"
    #std_title = f"hYield_STD_{bin_name}"
    wta_title = f"WTA_yield_{bin_name}"
    std_title = f"STD_yield_{bin_name}"

    return full_filepath, wta_title, std_title, bin_name


def openFiles(mult_bin):
    '''
    opens the yield histograms for a given multiplicity bin
    input: multiplicity bin
    outputs:
        - wta yield histogram
        - std yield histogram
        - wta Nassoc !
        - std Nassoc !
        - wta avg Nch
        - std avg Nch
    '''
    # 1. file path:
    filepath, wta_title, std_title, bin_name = pathfinder(mult_bin)

    if os.path.isfile(filepath) == False: #in case the file doesn't exist, i.e. this multiplicity bin was not used
        print(f"Error: Could not open file {filepath}")
        return None, None, 0, 0, 0, 0

    # 2. If it exists, open the file
    file = ROOT.TFile.Open(filepath, "READ")

    # 3. Open the WTA yield and detach it from the file directory
    hYield_wta = file.Get(wta_title)
    if hYield_wta:
        hYield_wta.SetDirectory(0)  # Keeps it alive in Python

    # 4. Open the STD yield and detach it
    hYield_std = file.Get(std_title)
    if hYield_std:
        hYield_std.SetDirectory(0)  # Keeps it alive in Python

    #get parameter values   
    obj_wta = file.Get(f"avg_Nch_WTA_{bin_name}")
    param_wta = ROOT.BindObject(obj_wta, ROOT.TParameter("double"))
    avg_Nch_wta = param_wta.GetVal()

    obj_std = file.Get(f"avg_Nch_STD_{bin_name}")
    param_std = ROOT.BindObject(obj_std, ROOT.TParameter("double"))
    avg_Nch_std = param_std.GetVal()

    # 5. Explicitly close the file (now completely safe to do!)
    file.Close()

    return hYield_wta, hYield_std, avg_Nch_wta, avg_Nch_std

In [25]:
#projection for |dEta| > 2 
# (note:jet frame cuts excluded eta star > 5)

eta_min = 2 #from prl paper, abs(delta eta star) should be greater than 2
eta_max = 4 #all the delta eta values should be less than this anyways i think

def project(hYield, suffix):
    '''
    Makes the 1d projection of the yield histograms
    input: yield histogram
    output: 1d projection
    '''
    # 1. Bins within delta eta star range
    bin_low = hYield.GetXaxis().FindBin(eta_min)
    bin_high = hYield.GetXaxis().FindBin(eta_max)
    
    # 2. doing the projection onto delta phi star
    h1D = hYield.Clone()
    h1D = h1D.ProjectionY(f"h1D_{suffix}", bin_low, bin_high)

    # 3. normalising by delta phi star bin width
    phi_bw = h1D.GetBinWidth(1)
    h1D.Scale(1.0 / phi_bw)

    # -------------------------------------------------------------------------
    # CRITICAL FIX: Convert raw X-axis bin sum to a Delta-Eta average
    # -------------------------------------------------------------------------
    #eta_bw = hYield.GetXaxis().GetBinWidth(1) # e.g., 0.1
    #eta_width = eta_max - eta_min             # 4 - 2 = 2.0
    
    #h1D.Scale(eta_bw / eta_width) 
    # -------------------------------------------------------------------------

    return h1D


In [26]:
#Fourier decomposition of the projection

#fourier fitting

#cosine series (param [0] should be N_assoc)
cosine_series = (
    "[0]/(2*TMath::Pi()) * (1 + 2*[1]*TMath::Cos(x) + 2*[2]*TMath::Cos(2*x) "
    "+ 2*[3]*TMath::Cos(3*x) + 2*[4]*TMath::Cos(4*x) + 2*[5]*TMath::Cos(5*x))"
)



def fourierFit(h1D, prefix, Nassoc = None):
    '''
    Applies a fourier fit to the 1d projection
    
    inputs: the 1d projection, and optional Nassoc
    output: the fourier coefficients and their errors. 

    modifications: the 1d histogram gets the fourier fit function put into it
    '''

    # root fit function
    fit_func = ROOT.TF1(f"fourier_fit_{prefix}", cosine_series, -0.5*math.pi , 1.5*math.pi)

    #seeding parameters
    if Nassoc is not None:
        fit_func.SetParameter(0, Nassoc)    #what it should be, based on the paper
    else:
        # "width" option calculates the true area under your 1D curve (height * 2pi)
        # This provides a mathematically perfect initial seed for parameter [0]
        fit_func.SetParameter(0, h1D.Integral("width"))
        
    fit_func.SetParameter(1, 0.1)   # v1 seed
    fit_func.SetParameter(2, 0.1)   # v2 seed (elliptic flow)
    fit_func.SetParameter(3, 0.1)   # v3 seed
    fit_func.SetParameter(4, 0.1)   # v4 seed
    fit_func.SetParameter(5, 0.1)   # v5 seed       all the seeds are 0.1 in the DrawFlow.C file so I'm just using that


    # EXECUTE THE FIT
    # ==========================================
    # "R" forces the fit range specified in the TF1 definition
    # "M" tells ROOT to search for better minimums (improves v2 precision)
    # "E" invokes the advanced Minos error estimation
    # "Q" keeps the terminal output quiet
    h1D.Fit(fit_func, "R M E Q")

    #extract parameters
    v1 = fit_func.GetParameter(1)
    v2 = fit_func.GetParameter(2)  # This is the fourier coefficient
    v3 = fit_func.GetParameter(3)

    #get error
    v1_err = fit_func.GetParError(1)
    v2_err = fit_func.GetParError(2)
    v3_err = fit_func.GetParError(3)

    #this bit is in the github but idk if i need it:
    # Apply the sqrt(2) scaling factor found in DrawVn's source code to conservatively account for Signal/Background statistical correlation
    v2_err_final = v2_err * math.sqrt(2)

    coeffs = [v1, v2, v3]
    errs = [v1_err, v2_err_final, v3_err]
    return coeffs, errs


In [27]:
def clean_hist(hist_obj):
    '''
    Safely deletes the C++ underlying memory of a detached ROOT object
    '''
    if hist_obj:
        hist_obj.Delete()

In [28]:
#modified from wta finder file

def saveProjections(mult_bin, h1D_wta, h1D_std):
    '''
    saves the projection histograms
    '''

    # Create the folder if it doesn't already exist
    os.makedirs(output_dir, exist_ok=True)

    # 2. Create the dynamic filename
    if len(mult_bin) == 2:
        bin_name = f"{mult_bin[0]}_{mult_bin[1]}"
    else:
        bin_name = f"{mult_bin[0]}_up"

    filename = f"Projection_Histograms_Mult_{bin_name}.root"

    # 3. Combine the folder path and the filename
    full_filepath = os.path.join(output_dir, filename)

    # 4. Open the ROOT file using the FULL path
    out_file = ROOT.TFile(full_filepath, "RECREATE")

    # 5. Rename and write the histograms
    h1D_wta.SetName(f"WTA_projection_{bin_name}")
    h1D_std.SetName(f"STD_projection_{bin_name}")

    h1D_wta.Write()
    h1D_std.Write()

    out_file.Close()

    print(f"Successfully saved projections to {full_filepath}")

In [29]:
#modified from gemini

# Ensure ROOT operates cleanly in Jupyter without popping up GUI windows
ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0) # Turn off the default statistical info boxes

def draw_projection_with_fit(h1D_wta, h1D_std, mult_bin):
    """
    Draws a 1D Delta-Phi projection histogram with its Fourier fit curve overlaid.
    """

    # make the file name
    if len(mult_bin) == 2:
        bin_name = f"{mult_bin[0]}_{mult_bin[1]}"
    else:
        bin_name = f"{mult_bin[0]}_up"
    filename = f"Projection_Mult_{bin_name}.pdf"
    save_path = os.path.join(output_dir, filename)

    canvas = ROOT.TCanvas("c_proj", "Projection Fit Validation", 800, 600)
    canvas.SetLeftMargin(0.15)
    canvas.SetBottomMargin(0.15)
    
    # General histogram aesthetics
    h1D_wta.SetMarkerStyle(20) # Filled circle
    h1D_wta.SetMarkerSize(1.1)
    h1D_wta.SetMarkerColor(ROOT.kBlue)
    h1D_wta.SetLineColor(ROOT.kBlue)
    h1D_wta.SetStats(0)

    h1D_std.SetMarkerStyle(25) 
    h1D_std.SetMarkerSize(1.1)
    h1D_std.SetMarkerColor(ROOT.kRed)
    h1D_std.SetLineColor(ROOT.kRed)
    h1D_std.SetStats(0)

    # 3. Dynamic Y-Axis Scaling
    # Find the global max and min so neither graph gets clipped
    max_y = max(h1D_wta.GetMaximum(), h1D_std.GetMaximum())
    min_y = min(h1D_wta.GetMinimum(), h1D_std.GetMinimum())
    
    # Add 25% headroom to the top for the legend
    h1D_wta.SetMaximum(max_y + 0.25 * abs(max_y))
    h1D_wta.SetMinimum(min_y - 0.10 * abs(min_y))
    
    # --- ROOT TLatex Title Fixes ---
    y_axis_title = "#frac{1}{N_{trig}} #frac{dN}{d#Delta#phi*}"
    
    if len(mult_bin) > 1:
        h1D_wta.SetTitle(f"Multiplicity Bin: {mult_bin[0]} #leq N_{{ch}} < {mult_bin[1]}; #Delta#phi*; {y_axis_title}")
    elif len(mult_bin) == 1:
        h1D_wta.SetTitle(f"Multiplicity Bin: {mult_bin[0]} #leq N_{{ch}}; #Delta#phi*; {y_axis_title}")

    # Draw 
    h1D_wta.Draw("PE") # P = Markers, E = Error bars
    h1D_std.Draw("PE SAME")
        
    # Fetch the function attached to the histogram to customize its display
    fit_curve_wta = h1D_wta.GetFunction("fourier_fit_wta")
    if fit_curve_wta:
        fit_curve_wta.SetLineColorAlpha(ROOT.kBlack, 0.5)
        fit_curve_wta.SetLineWidth(3)

    fit_curve_std = h1D_std.GetFunction("fourier_fit_std")
    if fit_curve_std:
        fit_curve_std.SetLineColorAlpha(ROOT.kBlack, 0.5)
        fit_curve_std.SetLineWidth(3)
    
    # 6. Add a standard Legend
    legend = ROOT.TLegend(0.45, 0.75, 0.88, 0.88)
    legend.SetBorderSize(0)
    legend.SetFillStyle(0) # Transparent background
    legend.SetTextSize(0.035)
    legend.AddEntry(h1D_wta, "Winner-Take-All (WTA)", "pe")
    legend.AddEntry(h1D_std, "Standard Axis", "pe")
    legend.Draw()
    
    # 7. Save directly as a PDF
    canvas.Update()
    canvas.SaveAs(save_path)
    canvas.Close()



def draw_summary(wta_x_vals, std_x_vals, wta_y, wta_err, std_y, std_err):
    """
    Creates a single summary graph comparing WTA vs STD v2 values.
    """
    wta_n_points = len(wta_x_vals)
    std_n_points = len(std_x_vals)
    
    # CRITICAL: Convert standard Python lists to C-compatible double arrays for PyROOT
    wta_x = array.array('d', wta_x_vals)
    std_x = array.array('d', std_x_vals)
    y_wta = array.array('d', wta_y)
    ey_wta = array.array('d', wta_err)
    y_std = array.array('d', std_y)
    ey_std = array.array('d', std_err)
    
    # Initialize TGraphErrors for both configurations
    gr_wta = ROOT.TGraphErrors(wta_n_points, wta_x, y_wta, 0, ey_wta)
    gr_std = ROOT.TGraphErrors(std_n_points, std_x, y_std, 0, ey_std)
    
    # Canvas initialization & grid configuration matching the reference
    canvas = ROOT.TCanvas("c_summary", "Anisotropy Coefficients vs Multiplicity", 750, 650)
    canvas.SetLeftMargin(0.15)
    canvas.SetBottomMargin(0.15)
    canvas.SetGrid()
    
    # Style Winner-Take-All Data points (Filled red markers)
    gr_wta.SetMarkerStyle(20)
    gr_wta.SetMarkerColor(ROOT.kBlue)
    gr_wta.SetLineColor(ROOT.kBlue)
    gr_wta.SetLineWidth(2)
    gr_wta.SetMarkerSize(1.3)
    
    # Style Standard Axis Data points (Open blue markers)
    gr_std.SetMarkerStyle(25) # Open square
    gr_std.SetMarkerColor(ROOT.kRed)
    gr_std.SetLineColor(ROOT.kRed)
    gr_std.SetLineWidth(2)
    gr_std.SetMarkerSize(1.3)
    
    # Wrap both inside a TMultiGraph to cleanly coordinate unified x and y scaling limits
    mg = ROOT.TMultiGraph()
    mg.Add(gr_std, "P")
    mg.Add(gr_wta, "P")
    mg.SetTitle("; Jet Multiplicity N_{ch}^{j}; Fourier Coefficient V_{#Delta 2}")
    mg.Draw("A") # "A" draws the bounding coordinate frame layout
    
    # Enforce strict axis scaling configurations
    mg.GetXaxis().SetLimits(0, 120)
    mg.GetHistogram().SetMinimum(-0.05)
    mg.GetHistogram().SetMaximum(0.15)
    
    mg.GetXaxis().SetTitleSize(0.045)
    mg.GetYaxis().SetTitleSize(0.045)
    mg.GetXaxis().SetLabelSize(0.04)
    mg.GetYaxis().SetLabelSize(0.04)
    
    # Render a baseline guide at v2 = 0
    line = ROOT.TLine(0, 0, 90, 0)
    line.SetLineStyle(2) # Dashed configuration
    line.SetLineColor(ROOT.kBlack)
    line.Draw()
    
    # Draw Legend using standard root bounding bounds
    legend = ROOT.TLegend(0.48, 0.74, 0.88, 0.88)
    legend.SetBorderSize(0)
    legend.SetFillStyle(0) # Transparent background
    legend.SetTextSize(0.035)
    legend.AddEntry(gr_wta, "Winner-Take-All (WTA) Axis", "pe")
    legend.AddEntry(gr_std, "Standard Axis", "pe")
    legend.Draw()
    
    # Add standardized scientific annotations
    #latex = ROOT.TLatex()
    #latex.SetNDC()
    #latex.SetTextSize(0.033)
    #latex.DrawLatex(0.18, 0.42, "#bf{CMS} Pythia 8 Simulation")
    #latex.DrawLatex(0.18, 0.37, "pp collisions #sqrt{s} = 13 TeV")
    #latex.DrawLatex(0.18, 0.32, "Jet p_{T} > 550 GeV/c, |#eta_{jet}| < 1.6")
    #latex.DrawLatex(0.18, 0.27, "Associated 0.3 < j_{T} < 3.0 GeV/c")
    #latex.DrawLatex(0.18, 0.22, "2.0 < |#Delta#eta^{*}| < 4.0 (Long-range)")
    
    canvas.Update()
    canvas.SaveAs("v2_summary_plot_ROOT.pdf")
    canvas.Close()

# running analysis:

In [31]:

#mult_bins = [ [0,20], [20,30], [30,40], [40,50], [50,60], [60,69], [69,79], [80] ]
#mult_bins = [ [0,20], [20,30], [30,40], [40,50], [50,60], [60,80], [80] ]

analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]


#initialising lists for the final plot
wta_x = []
wta_y = []
wta_y_err = []
std_x = []
std_y = []
std_y_err = []

for mult_bin in analysis_bins:
    hYield_wta , hYield_std, avg_Nch_wta, avg_Nch_std = openFiles(mult_bin)

    # If a file was missing, skip this iteration instead of crashing
    if hYield_wta is None or hYield_std is None:
        print(f"Skipping bin {mult_bin} because histograms could not be loaded.\n")
        continue

    #add avg Nch to x coordinate list
    wta_x.append(avg_Nch_wta)
    std_x.append(avg_Nch_std)

    #Extract v2 values
    #finding v2
    h1D_wta = project(hYield_wta, "wta")
    h1D_std = project(hYield_std, "std")

    #fit_wta = fourierFit(h1D_wta, Nassoc_wta)      <- if i do have Nassoc
    #fit_std = fourierFit(h1D_std, Nassoc_std)

    wta_fit, wta_errs = fourierFit(h1D_wta, "wta")      #<- if i don't have Nassoc
    std_fit, std_errs = fourierFit(h1D_std, "std")

    wta_v2 = [wta_fit[1], wta_errs[1]] #V2, error
    std_v2 = [std_fit[1], std_errs[1]] #V2, error

    # add to list
    wta_y.append(wta_v2[0])
    wta_y_err.append(wta_v2[1])
    std_y.append(std_v2[0])
    std_y_err.append(std_v2[1])

    #print results
    print("WTA fit for bin ",mult_bin," :", wta_fit, wta_errs)
    print("std fit for bin ",mult_bin," :", std_fit, std_errs)
    print("wta V2 for bin ",mult_bin," :", wta_v2[0], " ± ", wta_v2[1]) #fourier coeff, not elliptic flow. Elliptic flow is sqrt of this.
    print("std V2 for bin ",mult_bin," :", std_v2[0], " ± ", std_v2[1]) 
    print("wta avg Nch for bin ", mult_bin, " :", avg_Nch_wta)
    print("std avg Nch for bin ", mult_bin, " :", avg_Nch_std)

    #draw the graphs
    draw_projection_with_fit(h1D_wta, h1D_std, mult_bin)

    #clean out the histograms
    clean_hist(hYield_wta)
    clean_hist(hYield_std)
    clean_hist(h1D_wta)
    clean_hist(h1D_std)
    ROOT.gDirectory.Clear()


draw_summary(wta_x, std_x, wta_y, wta_y_err, std_y, std_y_err)

WTA fit for bin  [0, 25]  : [0.1, 0.1, 0.1] [0.0, 0.0, 0.0]
std fit for bin  [0, 25]  : [0.1, 0.1, 0.1] [0.0, 0.0, 0.0]
wta V2 for bin  [0, 25]  : 0.1  ±  0.0
std V2 for bin  [0, 25]  : 0.1  ±  0.0
wta avg Nch for bin  [0, 25]  : 0.0
std avg Nch for bin  [0, 25]  : 0.0
WTA fit for bin  [25, 36]  : [0.1, 0.1, 0.1] [0.0, 0.0, 0.0]
std fit for bin  [25, 36]  : [0.1, 0.1, 0.1] [0.0, 0.0, 0.0]
wta V2 for bin  [25, 36]  : 0.1  ±  0.0
std V2 for bin  [25, 36]  : 0.1  ±  0.0
wta avg Nch for bin  [25, 36]  : 0.0
std avg Nch for bin  [25, 36]  : 0.0
WTA fit for bin  [36, 48]  : [0.1, 0.1, 0.1] [0.0, 0.0, 0.0]
std fit for bin  [36, 48]  : [0.1, 0.1, 0.1] [0.0, 0.0, 0.0]
wta V2 for bin  [36, 48]  : 0.1  ±  0.0
std V2 for bin  [36, 48]  : 0.1  ±  0.0
wta avg Nch for bin  [36, 48]  : 0.0
std avg Nch for bin  [36, 48]  : 0.0
WTA fit for bin  [48, 60]  : [-0.1931129486131301, 0.16443509949349072, 0.06701661553660967] [0.07939967912747936, 0.09366561970092238, 0.05456407309945396]
std fit for bin  [48,

Warning in <Fit>: Fit data is empty 
Warning in <Fit>: Fit data is empty 
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_nch60/Projection_Mult_0_25.pdf has been created
Warning in <Fit>: Fit data is empty 
Warning in <Fit>: Fit data is empty 
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_nch60/Projection_Mult_25_36.pdf has been created
Warning in <Fit>: Fit data is empty 
Warning in <Fit>: Fit data is empty 
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_nch60/Projection_Mult_36_48.pdf has been created
Warning in <Fit>: Fit data is empty 
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_nch60/Projection_Mult_48_60.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjag

In [ ]:
#plot v2 vs multiplicity

In [ ]:
# for plotting fourier coeffs

wta_y_vals_0mb = []
wta_y_errs_0mb = []
std_y_vals_0mb = []
std_y_errs_0mb = []

wta_y_errs_3mb = []
wta_y_vals_3mb = []
std_y_vals_3mb = []
std_y_errs_3mb = []

high_bins = [ [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"

    print("0mb")
    h1D_wta_0mb, fit_wta_0mb, V2_delta_wta_0mb, v2_star_wta_0mb = extract_fourier_harmonics(wta_yields_0mb[bin_key], mult_bin, cluster_type="wta")
    h1D_std_0mb, fit_std_0mb, V2_delta_std_0mb, v2_star_std_0mb = extract_fourier_harmonics(std_yields_0mb[bin_key], mult_bin, cluster_type="std")

    print("3.0mb")
    h1D_wta_3mb, fit_wta_3mb, V2_delta_wta_3mb, v2_star_wta_3mb = extract_fourier_harmonics(wta_yields_3mb[bin_key], mult_bin, cluster_type="wta")
    h1D_std_3mb, fit_std_3mb, V2_delta_std_3mb, v2_star_std_3mb = extract_fourier_harmonics(std_yields_3mb[bin_key], mult_bin, cluster_type="std")

    wta_y_vals_0mb.append(V2_delta_wta_0mb[0])
    wta_y_errs_0mb.append(V2_delta_wta_0mb[1])

    wta_y_vals_3mb.append(V2_delta_wta_3mb[0])
    wta_y_errs_3mb.append(V2_delta_wta_3mb[1])

    std_y_vals_0mb.append(V2_delta_std_0mb[0])
    std_y_errs_0mb.append(V2_delta_std_0mb[1])

    std_y_vals_3mb.append(V2_delta_std_3mb[0])
    std_y_errs_3mb.append(V2_delta_std_3mb[1])